# 2 · Run the BB84 Experiment

Run BB84 across the slice from `01_setup_slice`: Bob listens, Alice sends photons through
the BMv2 quantum channel, classical sifting runs over the data-plane link. Results are
downloaded to `results/` and verified.

Wraps `deploy_fabric.run_bb84`. **Prerequisite:** `01_setup_slice` completed.

### At a glance
- **Purpose:** run one BB84 experiment across the slice and record the measured result.
- **Inputs:** `SLICE_NAME`, `SCENARIO` (must match notebook 1).
- **Outputs:** `results/fabric_{alice,bob}_results.json` (QBER, sifted bits, key rate) + a PASS/FAIL verification.
- **Runs on / runtime:** FABRIC JupyterHub; **~2–5 min**.
- **If something fails:** `No module named qne.cli` → the code didn't land on the nodes; re-run notebook 1's upload. Empty/zero results at long distance are expected (high fiber loss → few photons).

## Configuration (must match notebook 1)

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'
SCENARIO   = 'validation/scenarios/fabric_1km.yml'

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

# deploy_fabric.py holds the tested provisioning/run logic; the notebooks are
# thin wrappers around it so there is a single source of truth.
import deploy_fabric as df
from qne.config import ScenarioConfig

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

## Step 1 — Recover MACs and the data-plane IP from the slice
These were set in notebook 1; we read them back from the slice (a fresh kernel has no shared state).

In [ ]:
alice = slice_obj.get_node('alice')
bob = slice_obj.get_node('bob')
switch = slice_obj.get_node('switch')

alice_mac = alice.get_interface(network_name='net_alice_switch').get_mac()
bob_mac = bob.get_interface(network_name='net_switch_bob').get_mac()
sw_alice_mac = switch.get_interface(network_name='net_alice_switch').get_mac()
bob_data_ip = '10.10.1.2'   # assigned by setup_dataplane_ips() in notebook 1
print(f'Alice MAC {alice_mac} | Bob MAC {bob_mac} | switch Alice-side {sw_alice_mac}')
print(f'Bob data-plane IP (classical channel): {bob_data_ip}')

## Step 2 — Run BB84
Starts Bob then Alice, waits for completion, and downloads results to `results/fabric_*_results.json`.

In [ ]:
df.run_bb84(slice_obj, SCENARIO, alice_mac, bob_mac,
            sw_alice_mac=sw_alice_mac, bob_data_ip=bob_data_ip)

## Step 3 — Verify the run

In [ ]:
import json
checks = []
def record(name, ok, detail=''):
    checks.append(ok); print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f' — {detail}' if detail else ''))

rc, _ = switch.execute('pgrep -a simple_switch || true', quiet=True)
record('BMv2 simple_switch running', 'simple_switch' in rc)
for who in ['alice', 'bob']:
    p = PROJECT_DIR / 'results' / f'fabric_{who}_results.json'
    if not p.exists():
        record(f'{who} results downloaded', False); continue
    d = json.load(open(p)); record(f'{who} results downloaded', True)
    record(f'{who} sifted bits > 0', d.get('sifted_bits', 0) > 0, f"sifted={d.get('sifted_bits')}")
    q = d.get('qber', 1.0); record(f'{who} QBER plausible (< 0.11)', 0 <= q < 0.11, f'QBER={q:.4f}')
print('\nALL CHECKS PASSED' if all(checks) else '\nSOME CHECKS FAILED — see /tmp/{alice,bob}.log on the nodes and switch journalctl -u bmv2')

---
**Next:** `03_cross_validation` and `04_analysis`.

> To sweep parameters, change `SCENARIO` here **and** re-run notebook 1's Step 3
> (the loss-model threshold is installed on the switch from the scenario).